# FDG PET SUVR ANCOVA: CEUNIM vs Control

This notebook:
1. Loads `PET_CERMEP_CEUNIM_merged.csv` from the project `data/` folder.
2. Shows the 3-cohort structure in the merged file.
3. Compares **CEUNIM LC vs CEUNIM Control** ROI SUVR values using ANCOVA with `sexo` and `edad` as covariates.
4. Applies FDR correction across ROIs.
5. Creates publication-style boxplots for significant ROIs.

In [13]:
from pathlib import Path
import os
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests

pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid", context="talk")

In [ ]:
# Paths
merged_summary_csv_path = '../../../data/processed_pet/merged_summary_csv/'
filename_suvr_mean_harmonized = os.path.join(merged_summary_csv_path, 'summary_suvr_mean_harmonized.csv')
filename_suvr_cerebellum_harmonized = os.path.join(merged_summary_csv_path, 'summary_suvr_cerebellum_harmonized.csv')

output_path = '../../../results/pet/ancova_rois_cerebellum/'
os.makedirs(output_path, exist_ok=True)
plots_dir = os.path.join(output_path, "ancova_significant_rois")
os.makedirs(plots_dir, exist_ok=True)

In [15]:
# Load daata from the merged CSV file


df = pd.read_csv(filename_suvr_cerebellum_harmonized)

In [16]:
required_cols = ["group", "site", "sex", "age"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

group_site = np.select(
    [
        (df["site"].eq("CERMEP") & df["group"].eq("Controls")),
        (df["site"].eq("CEUNIM") & df["group"].eq("Controls")),
        (df["site"].eq("CEUNIM") & df["group"].eq("LC")),
    ],
    [
        "CERMEP_Control",
        "CEUNIM_Control",
        "CEUNIM_LC",
    ],
    default="Other",
)

summary = (
    df.assign(group_site=group_site)
    .groupby(["site", "group", "group_site"], dropna=False)
    .size()
    .reset_index(name="n")
)
print(summary)

     site     group      group_site   n
0  CERMEP  Controls  CERMEP_Control  37
1  CEUNIM  Controls  CEUNIM_Control  26
2  CEUNIM        LC       CEUNIM_LC  28


In [17]:
# CEUNIM LC vs CEUNIM Control
compare_df = df.loc[
    (df["site"].eq("CEUNIM")) & (df["group"].isin(["Controls", "LC"]))
].copy()

compare_df["sex"] = compare_df["sex"].astype("category")
compare_df["age"] = pd.to_numeric(compare_df["age"], errors="coerce")

excluded = {"ID", "group", "site", "sex", "age", "weight", "analysis_group"}
candidate_roi_cols = [c for c in compare_df.columns if c not in excluded]

roi_cols = []
for col in candidate_roi_cols:
    converted = pd.to_numeric(compare_df[col], errors="coerce")
    if converted.notna().sum() > 0:
        compare_df[col] = converted
        roi_cols.append(col)

print("N CEUNIM subjects for ANCOVA:")
print(compare_df["group"].value_counts(dropna=False))
print(f"Detected ROI SUVR columns: {len(roi_cols)}")

records = []

for roi in roi_cols:
    sub = compare_df[["group", "sex", "age", roi]].dropna().copy()

    if sub["group"].nunique() < 2:
        continue
    if sub["sex"].nunique() < 2:
        continue

    formula = f'Q("{roi}") ~ C(group) + C(sex) + age'
    model = ols(formula, data=sub).fit()
    anova_tbl = sm.stats.anova_lm(model, typ=2)

    group_term = "C(group)"
    if group_term not in anova_tbl.index:
        continue

    mean_control = sub.loc[sub["group"].eq("Controls"), roi].mean()
    mean_lc = sub.loc[sub["group"].eq("LC"), roi].mean()

    records.append(
        {
            "ROI": roi,
            "n": len(sub),
            "F_group": anova_tbl.loc[group_term, "F"],
            "p_group": anova_tbl.loc[group_term, "PR(>F)"],
            "mean_control": mean_control,
            "mean_lc": mean_lc,
            "delta_lc_minus_control": mean_lc - mean_control, 
            "direction": "hyper" if mean_lc > mean_control else "hypo"
        }
    )

results_df = pd.DataFrame(records).sort_values("p_group").reset_index(drop=True)

if len(results_df) > 0:
    rej, p_fdr, _, _ = multipletests(results_df["p_group"], alpha=0.05, method="fdr_bh")
    results_df["p_fdr"] = p_fdr
    results_df["significant_fdr"] = rej
else:
    results_df["p_fdr"] = pd.Series(dtype=float)
    results_df["significant_fdr"] = pd.Series(dtype=bool)

print(results_df.head(20))

output_filename_csv = os.path.join(output_path, "ancova_lc_vs_ceunim_controls_results.csv")
results_df.to_csv(output_filename_csv, index=False)

signif_df = results_df.loc[results_df["significant_fdr"].eq(True)].copy()
signif_df.to_csv(os.path.join(output_path, "ancova_lc_vs_ceunim_controls_significant_results_cerebellum.csv"), index=False)
print(f"Saved full ANCOVA table: {output_filename_csv}")
print(f"Significant ROIs (FDR < 0.05): {len(signif_df)}")

N CEUNIM subjects for ANCOVA:
group
LC          28
Controls    26
Name: count, dtype: int64
Detected ROI SUVR columns: 92
                                           ROI   n    F_group   p_group  \
0                CG-anterior-cingulate-gyrus-R  54  15.145025  0.000295   
1                   FL-lateral-orbital-gyrus-L  54  13.170451  0.000668   
2      TL-middle-and-inferior-temporal-gyrus-L  54  13.084089  0.000693   
3                    FL-middle-frontal-gyrus-L  54  11.204777  0.001556   
4                     PL-supramarginal-gyrus-L  54  10.461875  0.002162   
5                FL-subgenual-frontal-cortex-R  54  10.107638  0.002535   
6                  FL-inferior-frontal-gyrus-L  54  10.077608  0.002569   
7   TL-superior-temporal-gyrus-anterior-part-L  54  10.051487  0.002599   
8     TL-anterior-temporal-lobe-lateral-part-L  54   9.827604  0.002876   
9            FL-pre-subgenual-frontal-cortex-R  54   9.549900  0.003264   
10                 FL-superior-frontal-gyrus-R  54   

In [18]:
# CEUNIM LC vs CERMEP Control
compare_df = df.loc[
    (df["site"].eq("CEUNIM") & df["group"].eq("LC")) | (df["site"].eq("CERMEP") & df["group"].eq("Controls"))
].copy()

compare_df["sex"] = compare_df["sex"].astype("category")
compare_df["age"] = pd.to_numeric(compare_df["age"], errors="coerce")

excluded = {"ID", "group", "site", "sex", "age", "weight", "analysis_group"}
candidate_roi_cols = [c for c in compare_df.columns if c not in excluded]

roi_cols = []
for col in candidate_roi_cols:
    converted = pd.to_numeric(compare_df[col], errors="coerce")
    if converted.notna().sum() > 0:
        compare_df[col] = converted
        roi_cols.append(col)

print("N subjects for ANCOVA:")
print(compare_df["group"].value_counts(dropna=False))
print(f"Detected ROI SUVR columns: {len(roi_cols)}")

records = []

for roi in roi_cols:
    sub = compare_df[["group", "sex", "age", roi]].dropna().copy()

    if sub["group"].nunique() < 2:
        continue
    if sub["sex"].nunique() < 2:
        continue

    formula = f'Q("{roi}") ~ C(group) + C(sex) + age'
    model = ols(formula, data=sub).fit()
    anova_tbl = sm.stats.anova_lm(model, typ=2)

    group_term = "C(group)"
    if group_term not in anova_tbl.index:
        continue

    mean_control = sub.loc[sub["group"].eq("Controls"), roi].mean()
    mean_lc = sub.loc[sub["group"].eq("LC"), roi].mean()

    records.append(
        {
            "ROI": roi,
            "n": len(sub),
            "F_group": anova_tbl.loc[group_term, "F"],
            "p_group": anova_tbl.loc[group_term, "PR(>F)"],
            "mean_control": mean_control,
            "mean_lc": mean_lc,
            "delta_lc_minus_control": mean_lc - mean_control, 
            "direction": "hyper" if mean_lc > mean_control else "hypo"
        }
    )

results_df = pd.DataFrame(records).sort_values("p_group").reset_index(drop=True)
print(results_df)
if len(results_df) > 0:
    rej, p_fdr, _, _ = multipletests(results_df["p_group"], alpha=0.05, method="fdr_bh")
    results_df["p_fdr"] = p_fdr
    results_df["significant_fdr"] = rej
else:
    results_df["p_fdr"] = pd.Series(dtype=float)
    results_df["significant_fdr"] = pd.Series(dtype=bool)

results_df.head(20)

output_filename_csv = os.path.join(output_path, "ancova_lc_vs_cermep_controls_results.csv")
results_df.to_csv(output_filename_csv, index=False)

signif_df = results_df.loc[results_df["significant_fdr"].eq(True)].copy()
signif_df.to_csv(os.path.join(output_path, "ancova_lc_vs_cermep_controls_significant_results_cerebellum.csv"), index=False)
print(f"Saved full ANCOVA table: {output_filename_csv}")
print(f"Significant ROIs (FDR < 0.05): {len(signif_df)}")

N subjects for ANCOVA:
group
Controls    37
LC          28
Name: count, dtype: int64
Detected ROI SUVR columns: 92
                                            ROI   n   F_group   p_group  \
0                      PL-supramarginal-gyrus-L  65  3.230182  0.077243   
1                  TL-posterior-temporal-lobe-L  65  3.188153  0.079145   
2       TL-middle-and-inferior-temporal-gyrus-L  65  2.883929  0.094564   
3                    FL-lateral-orbital-gyrus-L  65  2.841375  0.096975   
4                 CG-anterior-cingulate-gyrus-R  65  2.747668  0.102531   
..                                          ...  ..       ...       ...   
87  Lateral-ventricle-excluding-temporal-horn-L  65  0.370352  0.545072   
88            Lateral-ventricle-temporal-horn-R  65  0.140859  0.708730   
89                                  OL-cuneus-L  65  0.135050  0.714525   
90        brainstem-excluding-substantia-nigra-  65  0.113304  0.737569   
91                                  OL-cuneus-R  65  0.03242

In [19]:
# CEUNIM LC vs All Controls
compare_df = df.loc[
    (df["site"].eq("CEUNIM") & df["group"].eq("LC")) | (df["group"].eq("Controls"))
].copy()

compare_df["sex"] = compare_df["sex"].astype("category")
compare_df["age"] = pd.to_numeric(compare_df["age"], errors="coerce")

excluded = {"ID", "group", "site", "sex", "age", "weight", "analysis_group"}
candidate_roi_cols = [c for c in compare_df.columns if c not in excluded]

roi_cols = []
for col in candidate_roi_cols:
    converted = pd.to_numeric(compare_df[col], errors="coerce")
    if converted.notna().sum() > 0:
        compare_df[col] = converted
        roi_cols.append(col)

print("N subjects for ANCOVA:")
print(compare_df["group"].value_counts(dropna=False))
print(f"Detected ROI SUVR columns: {len(roi_cols)}")

records = []

for roi in roi_cols:
    sub = compare_df[["group", "sex", "age", roi]].dropna().copy()

    if sub["group"].nunique() < 2:
        continue
    if sub["sex"].nunique() < 2:
        continue

    formula = f'Q("{roi}") ~ C(group) + C(sex) + age'
    model = ols(formula, data=sub).fit()
    anova_tbl = sm.stats.anova_lm(model, typ=2)

    group_term = "C(group)"
    if group_term not in anova_tbl.index:
        continue

    mean_control = sub.loc[sub["group"].eq("Controls"), roi].mean()
    mean_lc = sub.loc[sub["group"].eq("LC"), roi].mean()

    records.append(
        {
            "ROI": roi,
            "n": len(sub),
            "F_group": anova_tbl.loc[group_term, "F"],
            "p_group": anova_tbl.loc[group_term, "PR(>F)"],
            "mean_control": mean_control,
            "mean_lc": mean_lc,
            "delta_lc_minus_control": mean_lc - mean_control, 
            "direction": "hyper" if mean_lc > mean_control else "hypo"
        }
    )

results_df = pd.DataFrame(records).sort_values("p_group").reset_index(drop=True)
print(results_df)
if len(results_df) > 0:
    rej, p_fdr, _, _ = multipletests(results_df["p_group"], alpha=0.05, method="fdr_bh")
    results_df["p_fdr"] = p_fdr
    results_df["significant_fdr"] = rej
else:
    results_df["p_fdr"] = pd.Series(dtype=float)
    results_df["significant_fdr"] = pd.Series(dtype=bool)

results_df.head(20)

output_filename_csv = os.path.join(output_path, "ancova_lc_vs_all_controls_results.csv")
results_df.to_csv(output_filename_csv, index=False)

signif_df = results_df.loc[results_df["significant_fdr"].eq(True)].copy()
signif_df.to_csv(os.path.join(output_path, "ancova_lc_vs_all_controls_significant_results_cerebellum.csv"), index=False)
print(f"Saved full ANCOVA table: {output_filename_csv}")
print(f"Significant ROIs (FDR < 0.05): {len(signif_df)}")

N subjects for ANCOVA:
group
Controls    63
LC          28
Name: count, dtype: int64
Detected ROI SUVR columns: 92
                                      ROI   n   F_group   p_group  \
0                            cerebellum-R  91  9.056220  0.003424   
1           CG-anterior-cingulate-gyrus-R  91  8.977008  0.003561   
2                            cerebellum-L  91  8.277607  0.005048   
3       insula-anterior-inferior-cortex-L  91  6.504844  0.012508   
4           FL-subgenual-frontal-cortex-R  91  6.425797  0.013036   
..                                    ...  ..       ...       ...   
87  OL-lateral-remainder-occipital-lobe-R  91  0.410947  0.523175   
88                     OL-lingual-gyrus-R  91  0.367756  0.545809   
89      Lateral-ventricle-temporal-horn-R  91  0.061996  0.803956   
90                            OL-cuneus-L  91  0.011032  0.916590   
91                            OL-cuneus-R  91  0.009201  0.923803   

    mean_control   mean_lc  delta_lc_minus_control direc

In [20]:
plots_dir = repo_root / "results" / "pet_regional_analysis" / "plots" / "ancova_significant_rois"

plots_dir.mkdir(parents=True, exist_ok=True)



def clean_filename(name: str) -> str:

    return re.sub(r"[^A-Za-z0-9_.-]+", "_", name)





compare_df["analysis_group"] = compare_df["analysis_group"].replace(
    {"Control": "Control", "CEUNIM": "Long COVID"}
)

palette = {"Control": "#C5D3FF", "Long COVID": "#C573ED"}

if signif_df.empty:

    print("No significant ROIs after FDR correction. No plots generated.")

else:

    for _, row in signif_df.iterrows():

        roi = row["ROI"]

        sub = compare_df[["analysis_group", roi]].dropna().copy()



        fig, ax = plt.subplots(figsize=(6.4, 5.4), dpi=140)



        sns.boxplot(

            data=sub,

            x="analysis_group",

            y=roi,

            hue="analysis_group",

            order=["Control", "Long COVID"],

            hue_order=["Control", "Long COVID"],

            palette=palette,

            width=0.55,

            fliersize=0,

            linewidth=4,

            linecolor="black",

            dodge=False,

            legend=False,

            ax=ax,

        )



        sns.stripplot(

            data=sub,

            x="analysis_group",

            y=roi,

            order=["Control", "Long COVID"],

            hue_order=["Control", "Long COVID"],

            palette=palette,

            #color="#1A1A1A",
            
            edgecolor='black', linewidth=1,


            alpha=0.55,

            size=8,

            jitter=True,

            ax=ax,

        )



        ax.set_xlabel("Group", fontsize=12)

        ax.set_ylabel("Normalized FDG Uptake \n (whole-brain mean)", fontsize=12)

        ax.set_title(

            f"{roi}\nANCOVA group effect: p={row['p_group']:.3g}, FDR={row['p_fdr']:.3g}",

            fontsize=12,

            pad=10,

        )

        ax.spines["top"].set_visible(False)

        ax.spines["right"].set_visible(False)

        ax.grid(axis="y", alpha=0.55)



        plt.tight_layout()

        out_path = plots_dir / f"ancova_ceunim_vs_control_{clean_filename(roi)}.png"

        fig.savefig(out_path, dpi=300, bbox_inches="tight")

        plt.show()

        plt.close(fig)



    print(f"Saved {len(signif_df)} publication-style plots to: {plots_dir}")


NameError: name 'repo_root' is not defined

## Notes
- If you want to compare CEUNIM LC against **all controls** (CERMEP + CEUNIM), adjust the `compare_df` filter accordingly.
- You can switch from FDR to uncorrected threshold by using `p_group < 0.05` instead of `significant_fdr`.